In [2]:
from google.colab import files

print("Upload your domain_sentiment_data.tar file")
uploaded = files.upload()
print("Upload complete")

Upload your domain_sentiment_data.tar file


Saving negative.review copy to negative.review copy
Saving negative.review copy 2 to negative.review copy 2
Saving negative.review copy 3 to negative.review copy 3
Saving positive.review copy to positive.review copy
Saving positive.review copy 2 to positive.review copy 2
Saving positive.review copy 3 to positive.review copy 3
Saving negative.review to negative (1).review
Saving positive.review to positive (1).review
Saving unlabeled.review to unlabeled (1).review
Upload complete


In [3]:
import pandas as pd

# Read files
with open("positive.review", "r", encoding="utf-8") as f:
    positive_reviews = f.readlines()

with open("negative.review", "r", encoding="utf-8") as f:
    negative_reviews = f.readlines()

# Create DataFrame
df_positive = pd.DataFrame({
    "review": positive_reviews,
    "sentiment": 1  # Positive
})

df_negative = pd.DataFrame({
    "review": negative_reviews,
    "sentiment": 0  # Negative
})

# Combine
df = pd.concat([df_positive, df_negative], ignore_index=True)

# Shuffle
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df.head()

,review,sentiment
0,I. Mihaljevic\n,1
1,4 of 4\n,0
2,<reviewer_location>\n,1
3,Constantin Aldea\n,1
4,\n,1


In [6]:
print(df.shape)
print(df["sentiment"].value_counts())

(74471, 3)
sentiment
1    37371
0    37100
Name: count, dtype: int64


In [7]:
import re

def extract_reviews(filename):
    with open(filename, "r", encoding="utf-8") as f:
        content = f.read()

    # Extract text between <review_text> tags
    reviews = re.findall(r"<review_text>(.*?)</review_text>", content, re.DOTALL)

    return reviews

# Extract clean reviews
positive_reviews = extract_reviews("positive.review")
negative_reviews = extract_reviews("negative.review")

# Create DataFrame
import pandas as pd

df_positive = pd.DataFrame({
    "review": positive_reviews,
    "sentiment": 1
})

df_negative = pd.DataFrame({
    "review": negative_reviews,
    "sentiment": 0
})

df = pd.concat([df_positive, df_negative], ignore_index=True)

# Shuffle
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df.head()

,review,sentiment
0,\nI have extensive experience with wireless sy...,0
1,\nI set up a small security program with my Ma...,1
2,"\nBought these pretty cheap, from woot.com.\n\...",0
3,\nI got mine to work in about 30 min following...,1
4,\nWhile a nice product packed with great featu...,0


In [8]:
df.head(10)

,review,sentiment
0,\nI have extensive experience with wireless sy...,0
1,\nI set up a small security program with my Ma...,1
2,"\nBought these pretty cheap, from woot.com.\n\...",0
3,\nI got mine to work in about 30 min following...,1
4,\nWhile a nice product packed with great featu...,0
5,\nI recieved this unit 3 bussiness days after ...,0
6,\nGarmin iQue M5\n\nI'm very pleased with my l...,1
7,\n\nWhile spending hours trying to figure out ...,0
8,\nThese are great earbuds. I have alswys used ...,1
9,\nAfter 3 hours of banging my head against a w...,0


In [9]:
df["clean_review"] = df["review"].apply(clean_text)

In [10]:
df[["clean_review", "sentiment"]].head()

,clean_review,sentiment
0,i have extensive experience with wireless syst...,0
1,i set up a small security program with my maci...,1
2,bought these pretty cheap from woot com the go...,0
3,i got mine to work in about min following s hu...,1
4,while a nice product packed with great feature...,0


In [11]:
df = df[df["clean_review"].str.strip() != ""]

In [12]:
print(df.shape)
print(df["sentiment"].value_counts())

(2000, 3)
sentiment
0    1000
1    1000
Name: count, dtype: int64


In [13]:
from sklearn.model_selection import train_test_split

X = df["clean_review"]
y = df["sentiment"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Testing set:", X_test.shape)
print()
print("Training labels:")
print(y_train.value_counts())
print()
print("Validation labels:")
print(y_val.value_counts())
print()
print("Testing labels:")
print(y_test.value_counts())

Training set: (1400,)
Validation set: (300,)
Testing set: (300,)

Training labels:
sentiment
0    700
1    700
Name: count, dtype: int64

Validation labels:
sentiment
1    150
0    150
Name: count, dtype: int64

Testing labels:
sentiment
1    150
0    150
Name: count, dtype: int64


In [14]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words = 10000
max_length = 200

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

X_val_pad = pad_sequences(
    X_val_seq,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=max_length,
    padding="post",
    truncating="post"
)

print("X_train_pad shape:", X_train_pad.shape)
print("X_val_pad shape:", X_val_pad.shape)
print("X_test_pad shape:", X_test_pad.shape)

X_train_pad shape: (1400, 200)
X_val_pad shape: (300, 200)
X_test_pad shape: (300, 200)


In [15]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense, Dropout

model = Sequential([
    Embedding(input_dim=max_words, output_dim=64, input_length=max_length),
    GlobalAveragePooling1D(),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.summary()

history = model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=8,
    batch_size=32
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/8
44/44 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.4929 - loss: 0.6943 - val_accuracy: 0.5800 - val_loss: 0.6904
Epoch 2/8
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.5457 - loss: 0.6874 - val_accuracy: 0.5000 - val_loss: 0.6885
Epoch 3/8
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.5664 - loss: 0.6780 - val_accuracy: 0.6633 - val_loss: 0.6664
Epoch 4/8
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.6964 - loss: 0.6356 - val_accuracy: 0.7133 - val_loss: 0.6223
Epoch 5/8
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.7329 - loss: 0.5758 - val_accuracy: 0.6767 - val_loss: 0.5966
Epoch 6/8
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8257 - loss: 0.4739 - val_accuracy: 0.6833 - val_loss: 0.5808
Epoch 7/8
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8179 - loss: 0.4078 - val_accuracy: 0.6600 - val_loss: 0.5457
Epoch 8/8
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8557 - loss: 0.3692 - val_accuracy: 0.8133 - val_loss:

In [16]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

y_pred_prob = model.predict(X_test_pad)
y_pred = (y_pred_prob >= 0.5).astype(int).flatten()

print("Test accuracy:", accuracy_score(y_test, y_pred))
print()
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))
print()
print("Classification report:")
print(classification_report(y_test, y_pred, target_names=["Negative", "Positive"]))

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
Test accuracy: 0.8033333333333333

Confusion matrix:
[[117  33]
 [ 26 124]]

Classification report:
              precision    recall  f1-score   support

    Negative       0.82      0.78      0.80       150
    Positive       0.79      0.83      0.81       150

    accuracy                           0.80       300
   macro avg       0.80      0.80      0.80       300
weighted avg       0.80      0.80      0.80       300



In [17]:
def predict_review(review):
    cleaned = clean_text(review)
    sequence = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(
        sequence,
        maxlen=max_length,
        padding="post",
        truncating="post"
    )

    prediction = model.predict(padded, verbose=0)[0][0]

    if prediction >= 0.5:
        sentiment = "Positive"
    else:
        sentiment = "Negative"

    return sentiment, prediction

test_reviews = [
    "This is the best product I have ever bought!",
    "Everyone should buy this product. I love it. It works so well. Incredible!",
    "This product is terrible and I hate it.",
    "Worst purchase ever. Completely useless.",
    "I love that I hate this product. It is terrible."
]

for review in test_reviews:
    sentiment, score = predict_review(review)
    print(review)
    print("Prediction:", sentiment)
    print("Score:", score)
    print()

This is the best product I have ever bought!
Prediction: Positive
Score: 0.6061721

Everyone should buy this product. I love it. It works so well. Incredible!
Prediction: Positive
Score: 0.585405

This product is terrible and I hate it.
Prediction: Negative
Score: 0.47492203

Worst purchase ever. Completely useless.
Prediction: Negative
Score: 0.4580279

I love that I hate this product. It is terrible.
Prediction: Negative
Score: 0.49248368



In [18]:
import pickle

model.save("my_model.keras")

with open("tokenizer.pkl", "wb") as handle:
    pickle.dump(tokenizer, handle)

print("Saved new my_model.keras and tokenizer.pkl")

Saved new my_model.keras and tokenizer.pkl


In [19]:
from google.colab import files

files.download("my_model.keras")
files.download("tokenizer.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
from sklearn.model_selection import train_test_split

X = df["clean_review"]
y = df["sentiment"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Testing set:", X_test.shape)
print()
print("Training labels:")
print(y_train.value_counts())
print()
print("Validation labels:")
print(y_val.value_counts())
print()
print("Testing labels:")
print(y_test.value_counts())

Training set: (1400,)
Validation set: (300,)
Testing set: (300,)

Training labels:
sentiment
0    700
1    700
Name: count, dtype: int64

Validation labels:
sentiment
1    150
0    150
Name: count, dtype: int64

Testing labels:
sentiment
1    150
0    150
Name: count, dtype: int64


In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Build pipeline
sentiment_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=10000,
        ngram_range=(1, 2),
        stop_words="english"
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

# Train model
sentiment_model.fit(X_train, y_train)

# Predict
y_pred = sentiment_model.predict(X_test)

# Evaluate
print("Test accuracy:", accuracy_score(y_test, y_pred))
print()
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))
print()
print("Classification report:")
print(classification_report(y_test, y_pred, target_names=["Negative", "Positive"]))

Test accuracy: 0.83

Confusion matrix:
[[117  33]
 [ 18 132]]

Classification report:
              precision    recall  f1-score   support

    Negative       0.87      0.78      0.82       150
    Positive       0.80      0.88      0.84       150

    accuracy                           0.83       300
   macro avg       0.83      0.83      0.83       300
weighted avg       0.83      0.83      0.83       300



In [27]:
import pickle

with open("sentiment_model.pkl", "wb") as f:
    pickle.dump(sentiment_model, f)

print("Saved sentiment_model.pkl")

Saved sentiment_model.pkl


In [28]:
from google.colab import files

files.download("sentiment_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>